In [1]:
RECIPE_PROMPT = """
你是一位专业的中餐厨师和营养师。请根据提供的现有食材和约束条件（如忌口、饮食偏好、烹饪时间等），推荐 3 道适合的中式菜谱。

返回 JSON 格式如下：

{
  "recipes": [
    {
      "name": "菜谱名称",
      "cuisine": "菜系（如：川菜、粤菜、湘菜、鲁菜、苏菜、浙菜、闽菜、徽菜、家常菜等）",
      "taste": "口味（如：麻辣、清淡、酸甜、咸鲜、香辣等）",
      "difficulty": "难度（简单 / 中等 / 困难）",
      "prep_time": "准备时间（X 分钟）",
      "cook_time": "烹饪时间（X 分钟）",
      "servings": N,
      "ingredients_needed": [
        "所需食材（用量）"
      ],
      "missing_ingredients": [
        "建议补充的食材（可选，没有则返回空数组）"
      ],
      "seasonings": [
        "调料（用量）"
      ],
      "instructions": [
        "步骤1：……",
        "步骤2：……"
      ],
      "nutrition_per_serving": {
        "calories": N,
        "protein": "Xg",
        "carbs": "Xg",
        "fat": "Xg"
      },
      "tips": "烹饪技巧或主厨建议"
    }
  ],
  "recommended": "最推荐的菜谱名称及推荐理由"
}

要求：
1. 优先使用用户已有食材，尽量减少需要额外购买的食材。
2. 推荐符合中国家庭日常烹饪习惯的菜品。
3. 所需调料默认为中国家庭常见调料（如食盐、生抽、老抽、料酒、蚝油、白糖、醋、食用油等），无需列入缺少食材；若需要特殊调料，再列入 `missing_ingredients`。
4. 烹饪步骤应简洁清晰，适合家庭厨房操作。
5. 营养信息为每人份的估算值。
6. `recommended` 应选择最符合用户现有食材和约束条件的一道菜，并简要说明推荐原因。
7. 仅返回合法的 JSON，不要输出任何解释、说明或 Markdown 代码块。
"""

In [2]:
import re
import json

def parse_json_response(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)
    return json.loads(cleaned)

In [ ]:
def display_recipe(recipe: dict):
    print(f"\n🍽️  {recipe.get('name', 'Recipe')} ({recipe.get('cuisine', 'N/A')})")
    print(f"⏱️  Prep: {recipe.get('prep_time', 'N/A')} | Cook: {recipe.get('cook_time', 'N/A')} | Difficulty: {recipe.get('difficulty', 'N/A')}")
    print(f"👥 Serves: {recipe.get('servings', 'N/A')}")
    print(f"\n📝 Ingredients:")
    for ing in recipe.get("ingredients_needed", []):
        print(f"  • {ing}")
    if recipe.get("missing_ingredients"):
        print(f"\n➕ Optional additions: {', '.join(recipe['missing_ingredients'])}")
    print(f"\n👨‍🍳 Instructions:")
    for i, step in enumerate(recipe.get("instructions", []), 1):
        print(f"  {i}. {step}")
    n = recipe.get("nutrition_per_serving", {})
    if n:
        print(f"\n🥗 Nutrition: {n.get('calories', '?')} cal | Protein: {n.get('protein', '?')} | Carbs: {n.get('carbs', '?')} | Fat: {n.get('fat', '?')}")
    if recipe.get("tips"):
        print(f"\n💡 Chef's tip: {recipe['tips']}")

In [22]:
from typing import cast
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

from dotenv import load_dotenv

load_dotenv()

import os

model = os.getenv('MODEL_NAME', '')

llm = ChatOpenAI(model=model, temperature=0.2, extra_body={'enable_thinking': False})

def get_recipes(ingredients: list[str], diet: str, time_limit: int, servings: int, stream: bool):
    constraints = []
    if diet:
        constraints.append(f"饮食限制：{diet}")
    if time_limit:
        constraints.append(f"最长总用时：{time_limit} 分钟")
    if servings:
        constraints.append(f"所需份数：{servings}")

    messages = [
        SystemMessage(content=RECIPE_PROMPT),
        HumanMessage(
            content=(
                f"现有食材：{', '.join(ingredients)}\n\n"
                f"限制条件：{chr(10).join(constraints) if constraints else '无'}"
            )
        ),
    ]
    if (not stream):
        response = llm.invoke(input=messages)
        return parse_json_response(text=cast(str, response.content))
    else:
        return llm.stream(input=messages)

In [ ]:
from langchain_core.messages import AIMessageChunk

for chunk in get_recipes(ingredients=["鸡蛋", "牛奶", "面粉"], diet="无", time_limit=30, servings=2, stream=True):
    if isinstance(chunk, AIMessageChunk):
        print(chunk.content, end="", flush=True)

{
  "recipes": [
    {
      "name": "牛奶鸡蛋软饼",
      "cuisine": "家常菜",
      "taste": "咸鲜/清淡",
      "difficulty": "简单",
      "prep_time": "10",
      "cook_time": "15",
      "servings": 2,
      "ingredients_needed": [
        "面粉 150g",
        "鸡蛋 2个",
        "牛奶 150ml",
        "食用油 适量（煎制用）"
      ],
      "missing_ingredients": [],
      "seasonings": [
        "食盐 3g",
        "胡椒粉 少许（可选，若无可不加）"
      ],
      "instructions": [
        "步骤1：将面粉放入大碗中，打入两个鸡蛋，加入少许盐和胡椒粉。",
        "步骤2：分次倒入牛奶，边倒边搅拌，直至形成无颗粒、流动性适中的面糊（类似酸奶质地）。",
        "步骤3：平底锅刷一层薄油，小火预热。舀一勺面糊倒入锅中，迅速转动锅子使面糊铺平。",
        "步骤4：中小火煎至表面凝固并出现小气泡，边缘微黄时翻面，再煎30秒至两面金黄即可出锅。",
        "步骤5：重复上述步骤直到面糊用完。可搭配番茄酱或蜂蜜食用。"
      ],
      "nutrition_per_serving": {
        "calories": 380,
        "protein": "14g",
        "carbs": "52g",
        "fat": "12g"
      },
      "tips": "面糊不要太稠，否则饼会硬；也不要太稀，否则不易成型。全程保持小火，避免外焦里生。"
    },
    {
      "name": "牛奶鸡蛋羹",
      "cuisine": "粤菜/家常",
      "taste": "清淡/奶香",
      "difficulty": "简单",


In [23]:
result = get_recipes(ingredients=["鸡蛋", "牛奶", "面粉"], diet="无", time_limit=30, servings=2, stream=False)

if isinstance(result, dict):
    print("\n" + "=" * 60)
    recipes = result.get("recipes", [])
    recommended = result.get("recommended") or (recipes[0].get("name", "N/A") if recipes else "N/A")
    print("✅ RECOMMENDED:", recommended)
    print("=" * 60)

    for recipe in recipes:
        display_recipe(recipe)
        print("\n" + "-" * 40)



✅ RECOMMENDED: 牛奶鸡蛋软饼。推荐理由：完全使用现有食材，无需额外购买任何调料或辅料；烹饪过程仅需煎制，耗时短（约15-20分钟），符合30分钟时限；操作简单成功率高，且营养均衡，适合作为早餐或简餐。

🍽️  牛奶鸡蛋软饼 (家常菜)
⏱️  Prep: 10 | Cook: 15 | Difficulty: 简单
👥 Serves: 2

📝 Ingredients:
  • 面粉 150g
  • 鸡蛋 2个
  • 牛奶 150ml
  • 食用油 适量（用于煎制）

👨‍🍳 Instructions:
  1. 步骤1：将面粉放入大碗中，打入两个鸡蛋，加入少许盐。
  2. 步骤2：分次倒入牛奶和清水，边倒边搅拌，直到形成无颗粒、流动性适中的面糊（类似酸奶质地）。
  3. 步骤3：平底锅预热，刷一层薄油，舀入一勺面糊，晃动锅子使面糊铺平。
  4. 步骤4：中小火煎至表面凝固并出现小气泡，翻面继续煎另一面，直至两面金黄熟透即可出锅。
  5. 步骤5：重复步骤3-4，直到面糊用完。可搭配番茄酱或蜂蜜食用。

🥗 Nutrition: 380 cal | Protein: 14g | Carbs: 55g | Fat: 12g

💡 Chef's tip: 面糊不要太稀，否则容易破；也不要太稠，否则口感硬。如果家里有葱花或火腿丁，可以加进去增加风味。

----------------------------------------

🍽️  家常鸡蛋炒牛奶 (粤菜改良版)
⏱️  Prep: 5 | Cook: 10 | Difficulty: 中等
👥 Serves: 2

📝 Ingredients:
  • 鸡蛋 3个
  • 牛奶 200ml
  • 面粉 10g（作为增稠剂，可选，若没有可用玉米淀粉代替，但用户只有面粉）

👨‍🍳 Instructions:
  1. 步骤1：将鸡蛋打散，加入牛奶、白糖和少许盐，搅拌均匀。
  2. 步骤2：为了增加凝固性，可将10g面粉用少量冷水化开，倒入蛋奶液中搅匀（防止牛奶分离，使成品更嫩滑）。
  3. 步骤3：平底锅烧热，倒入少许油润锅后倒出，利用余温倒入蛋奶液（低温慢炒是关键）。
  4. 步骤4：用铲子轻轻从边缘向中间推炒，保持小火，直到蛋奶液大部分凝固但仍保持湿润嫩滑状态即可关火